# Modules, Packages and Virtual Environments

Every notebook so far has quietly started with a line like `import csv` or `import os`, and we never stopped to ask what that line actually does.

That is the gap this chapter closes. By the end you will be able to split your code across several files, reuse it, install code other people wrote, and set up a project so that it still runs on somebody else's computer.

**What we will learn:**

1. What a module really is, and the three ways to import one
2. What `import` actually does behind the scenes
3. Writing your own module, and importing it
4. `if __name__ == "__main__"` and why almost every Python file has it
5. Packages: organising many modules into one folder
6. The standard library: the modules you get for free
7. Third-party packages and `pip`
8. Virtual environments, and why every real project has one
9. How a real Python project is laid out

---
# 1. What Is a Module?

A **module is just a Python file**. That is the whole idea.

Any file ending in `.py` is a module, and `import` is how you use the code inside one file from another file. Python ships with hundreds of these files ready to use, which is why you have been able to write `import csv` since chapter 10 without ever creating a `csv.py` yourself.

You have already used more modules than you might realise:

In [1]:
import math

math.sqrt(144)

12.0

One line, and we got a square-root function we did not write.

A module is not special syntax — once imported, it is an **object** like any other, and the names inside it are its attributes. That is worth seeing directly, because it explains why `dir()` and `help()` work on modules the same way they work on anything else:

In [2]:
print(type(math))          # a module really is just an object
print(math.__name__)       # every module knows its own name
print("sqrt" in dir(math))  # dir() lists the names a module contains

<class 'module'>
math
True


Those two functions are how you explore an unfamiliar module without leaving your editor:

- **`dir(module)`** lists every name inside it.
- **`help(module.thing)`** prints the documentation for one of them.

Try `help(math.floor)` in a cell if you want to see what that looks like. It is the fastest way to answer "what does this actually do?" without searching the web.

Here are the modules that have already appeared in this course, and what each one gave us:

| Module | Chapter | What we used it for |
|---|---|---|
| `os` | 10 | Creating folders, listing directories |
| `pathlib` | 10 | The modern `Path` object |
| `csv` | 10 | Reading and writing spreadsheet files |
| `json` | 10 | Saving and loading structured data |
| `functools` | 7 | `reduce()` |
| `collections` | 10 | `defaultdict` for grouping |
| `time` | 7 | Timing how long code takes |
| `keyword` | 1 | Listing Python's reserved words |

None of that code is built into the language itself. It all lives in files that ship with Python, and `import` is the door to them.

---
## The Three Ways to Import

There are three import styles you will see constantly. They differ only in **what name you end up with**.

**Style 1: `import module`** — you get the module, and reach inside it with a dot.

In [3]:
import math

print(math.pi)
print(math.floor(9.7))

3.141592653589793
9


**Style 2: `from module import name`** — you get one specific thing, with no dot needed.

In [4]:
from math import pi, floor

print(pi)
print(floor(9.7))

3.141592653589793
9


**Style 3: `import module as alias`** — same as style 1, but you rename it.

This is not just laziness. Some library names are long, and the data science world has settled on standard aliases that you will see in every tutorial and every job: `import pandas as pd`, `import numpy as np`.

In [5]:
import statistics as stats

stats.mean([10, 20, 30, 40])

25

**And a fourth style you should avoid: `from module import *`**

This imports *everything* from the module directly into your file. It looks convenient and it causes real bugs:

```python
from math import *      # don't do this
```

The problem is that you can no longer tell where anything came from. If two modules both define `sqrt`, the second import silently wins and you get no warning. Worse, if you wrote your own function called `floor`, `from math import *` will quietly overwrite it.

Here is that bug, in four lines. We write our own `floor()` that chops the decimals off a price, and then an innocent-looking `import *` silently replaces it:

In [6]:
def floor(price):
    return int(price)           # our version: just chop the decimals off

print("our floor:", floor(-9.99))

from math import *              # looks harmless -- but math has a floor() too

print("after import *:", floor(-9.99))

our floor: -9
after import *: -10


Same function call, two different answers, and nothing warned us. `int()` chops toward zero and gives `-9`; `math.floor()` rounds *down* and gives `-10`.

In a four-line cell that is obvious. In a 300-line file where the `import *` sits at the top and the call sits at the bottom, it is a genuinely hard bug to find.

Be explicit. Your future self reading the code needs to know where each name came from.

**So which style should you use?**

| Situation | Use |
|---|---|
| You need several things from a module | `import module`, then `module.thing` |
| You use one or two names, a lot | `from module import name` |
| The library has a conventional short name | `import pandas as pd` |
| Two modules both define the same name | `import module` — the dot keeps them apart |
| Ever | not `import *` |

When in doubt, `import module` is the safe default: it is always obvious where a name came from.

---
## What `import` Actually Does

When you write `import math`, Python does three things:

1. **Finds the file.** It searches a list of folders, in order, until it finds one called `math`.
2. **Runs it, top to bottom, once.** Every line in the module executes — function definitions get defined, and any code sitting at the top level actually runs.
3. **Caches it.** The result goes into `sys.modules` so that importing it again is instant, and the file is *not* run a second time.

That third point surprises people, so let us prove it.

In [7]:
import sys

"math" in sys.modules

True

That caching is not just a speed trick. Because a module runs only once no matter how many files import it, any setup it does at the top level — opening a connection, loading a config file, building a lookup table — happens a single time and is then shared by everything that imports it.

It also means that if you edit a module while Python is still running, your changes are **not** picked up by a fresh `import`. That is the number one confusion when working in notebooks: you fix a bug in your `.py` file, re-run the import, and nothing changes. The fix is to restart the kernel (or use `importlib.reload`).

The list of folders Python searches is `sys.path`. The first entry is the folder your script lives in, which is why a module sitting next to your file "just works" with no setup at all.

We will use this in a moment, because we are about to put a module in a subfolder.

---
# 2. Writing Your Own Module

Once a project grows past a couple of hundred lines, keeping everything in one file stops being comfortable. The fix is to move related functions into their own file and import them.

A useful rule of thumb for *when* to split: **group by what the code is about, not by what type of thing it is.** A file called `text_utils.py` holding every text-related function is easy to find things in. A file called `functions.py` holding everything is just the original problem with extra steps.

To keep this repository tidy, everything we create in this chapter goes into a folder called `module_demo/`. We will start by clearing it out, so this notebook produces the same result every time you run it.

In [8]:
import shutil
from pathlib import Path

demo = Path("module_demo")
if demo.exists():
    shutil.rmtree(demo)          # start clean, so re-running gives the same output
demo.mkdir()

print(f"{demo}/ is ready")

module_demo/ is ready


Now we write an actual module. The `%%writefile` line at the top of the next cell is a Jupyter shortcut that saves the rest of the cell to a file instead of running it — a convenient way to create a `.py` file from inside a notebook.

Everything below that first line is ordinary Python, exactly what you would type into `text_utils.py` in a normal editor.

In [9]:
%%writefile module_demo/text_utils.py
# Small text helpers, the kind every project accumulates.

def clean_title(title):
    # "  the   MATRIX  " -> "The Matrix"
    return " ".join(title.split()).title()


def slugify(title):
    # "The Matrix Reloaded" -> "the-matrix-reloaded"
    return "-".join(clean_title(title).lower().split())


def shorten(text, limit=30):
    # Cut long text down for a preview, without cutting mid-word.
    if len(text) <= limit:
        return text
    return text[:limit].rsplit(" ", 1)[0] + "..."


Writing module_demo/text_utils.py


That file now exists on disk. To import it, Python has to be able to find it — and `module_demo/` is a subfolder, not the folder we are working in.

So we add it to `sys.path`, the list of places Python searches:

In [10]:
import sys

sys.path.insert(0, "module_demo")

import text_utils

print(text_utils.clean_title("  the   MATRIX  "))
print(text_utils.slugify("The Matrix Reloaded"))
print(text_utils.shorten("Python makes text processing genuinely pleasant", 30))

The Matrix
the-matrix-reloaded
Python makes text processing...


**A note on that `sys.path.insert` line:** in a normal project you would not need it. If `text_utils.py` sat next to the file importing it, the import would work with no setup, because a script's own folder is always searched first.

We only need it here because we deliberately put the module in a subfolder to keep this repository organised.

Your own modules obey exactly the same import rules as `math` did, so the `from ... import ...` form works here too:

In [11]:
from text_utils import slugify, shorten

print(slugify("Breaking Bad Season Five"))
print(shorten("A very long description that needs trimming for the card", 25))

breaking-bad-season-five
A very long description...


There is nothing special about the standard library. `math` is a file someone wrote and shipped with Python; `text_utils.py` is a file you wrote and put in a folder. Python treats them identically.

Every module knows its own name, in a variable called `__name__` that Python sets automatically:

In [12]:
text_utils.__name__

'text_utils'

---
# 3. `if __name__ == "__main__"`

Here is the single most common idiom in Python, and it exists because of something we noted earlier: **importing a module runs every line in it**.

That is fine for function definitions. It is a problem for anything else. Watch what happens when a module has a stray `print` at the top level:

In [13]:
%%writefile module_demo/greet.py
def welcome(name):
    return f"Welcome back, {name}!"

# This line is NOT inside a function, so it runs the moment the file is imported.
print("*** greet.py is running ***")


Writing module_demo/greet.py


In [14]:
import greet

print(greet.welcome("Shikhar"))

*** greet.py is running ***
Welcome back, Shikhar!


Notice the `*** greet.py is running ***` line. We only wanted the `welcome` function, but importing the module ran its print statement too.

And because Python caches modules, importing it a second time does nothing at all:

In [15]:
import greet          # already in sys.modules, so the file does not run again

print("...and this time the module did not run.")

...and this time the module did not run.


So how do you write a file that can be **both** a reusable module *and* a script you can run directly?

That is exactly what `__name__` is for. It is a variable Python sets automatically in every file — including this notebook. Run this and see what your own code is called:

In [16]:
print(__name__)

__main__


`__main__` means *"this is the program being run"*, which is exactly what a notebook is.

Python sets `__name__` to:

- `"__main__"` when the file is **run directly**, and
- **the module's name** when the file is **imported**.

Checking it lets you mark off a block that should only run when the file is executed as a program:

In [17]:
%%writefile module_demo/report.py
def summarise(titles):
    return f"{len(titles)} titles, longest is '{max(titles, key=len)}'"


if __name__ == "__main__":
    # Only runs when you execute:  python3 report.py
    # Skipped entirely when another file does:  import report
    demo_titles = ["Stranger Things", "Wednesday", "The Crown"]
    print(summarise(demo_titles))


Writing module_demo/report.py


Import it, and the guarded block stays quiet:

In [18]:
import report

print(report.summarise(["Dune", "Arrival"]))

2 titles, longest is 'Arrival'


Run the very same file as a script, and the block executes:

In [19]:
!python3 module_demo/report.py

3 titles, longest is 'Stranger Things'


Same file. Two behaviours. That is why you will see `if __name__ == "__main__":` at the bottom of almost every Python file you ever read.

The practical rule: **put your functions and classes at the top level, and put anything that actually *does* something inside the guard.**

---
# 4. Packages: Folders of Modules

One file is a module. **A folder of modules is a package.**

When a project has fifteen utility functions, splitting them across `text_utils.py`, `date_utils.py` and `money_utils.py` helps — but now your project folder is cluttered. A package groups them under one name.

The folder becomes a package by containing a file called `__init__.py`. It can be completely empty; its presence is the signal.

In [20]:
Path("module_demo/netflix").mkdir()

print("created module_demo/netflix/")

created module_demo/netflix/


In [21]:
%%writefile module_demo/netflix/__init__.py
# This file marks the folder as a package.
# It can be empty. Code placed here runs when the package is imported.


Writing module_demo/netflix/__init__.py


In [22]:
%%writefile module_demo/netflix/catalog.py
def format_runtime(minutes):
    hours, mins = divmod(minutes, 60)
    if hours == 0:
        return f"{mins}m"
    return f"{hours}h {mins}m"


Writing module_demo/netflix/catalog.py


In [23]:
%%writefile module_demo/netflix/ratings.py
def average(scores):
    if not scores:
        return 0.0
    return round(sum(scores) / len(scores), 2)


def label(score):
    if score >= 4.5:
        return "Critically acclaimed"
    elif score >= 3.5:
        return "Well received"
    return "Mixed reviews"


Writing module_demo/netflix/ratings.py


Now the whole folder imports as one unit, and the dot separates package from module:

In [24]:
from netflix import catalog, ratings

print(catalog.format_runtime(154))
print(catalog.format_runtime(47))

scores = [4.8, 4.6, 4.9, 4.4]
print(ratings.average(scores), "->", ratings.label(ratings.average(scores)))

2h 34m
47m
4.67 -> Critically acclaimed


You can also reach straight for one function, which is the style you will see most often:

In [25]:
from netflix.ratings import label

label(3.9)

'Well received'

**What goes in `__init__.py`?**

Ours is empty, and empty is a perfectly normal choice. But it runs when the package is imported, which makes it the place to give your package a **friendly front door**: instead of making users remember which module each function lives in, you re-export the useful ones.

That is why `import pandas as pd` gives you `pd.DataFrame` directly, even though `DataFrame` is defined several files deep.

In [26]:
%%writefile module_demo/netflix/__init__.py
# Pull the useful names up to the package level, so users can write
#     from netflix import average
# instead of having to know that average() lives in netflix/ratings.py
from netflix.ratings import average, label
from netflix.catalog import format_runtime


Overwriting module_demo/netflix/__init__.py


We have to run this from a **fresh Python process** to see the effect: this notebook already imported `netflix`, and as we saw earlier, Python caches modules and will not re-run the file.

Note that this script needs no `sys.path` line — it lives inside `module_demo/`, right next to the package it imports.

In [27]:
%%writefile module_demo/front_door.py
import netflix          # not netflix.ratings -- the package hands us the names directly

print(netflix.format_runtime(96))
print(netflix.label(4.7))


Writing module_demo/front_door.py


In [28]:
!python3 module_demo/front_door.py

1h 36m
Critically acclaimed


Same functions, shorter import. The package now presents one clean surface, and you are free to reorganise the files behind it without breaking anyone's code — which is the real reason libraries do this.

So the vocabulary, in one line each:

| Term | What it is |
|---|---|
| **Module** | A single `.py` file |
| **Package** | A folder of modules, marked by `__init__.py` |
| **Library** | An informal word for a package (or group of packages) you install and use |
| **Standard library** | The packages that ship with Python itself |

---
# 5. The Standard Library: Batteries Included

Python's slogan for years was *"batteries included"*: a large collection of useful modules ships with the language, so a fresh install can already read CSVs, parse dates, do maths and talk to the file system.

You have met eight of them already. Here are a few more worth knowing before you go further:

In [29]:
import random

random.seed(42)                     # fixes the "randomness" so results repeat

print(random.randint(1, 100))
print(random.choice(["Netflix", "Spotify", "Amazon"]))

82
Netflix


**A note on `random.seed()`:** random numbers on a computer are not truly random — they come from a formula with a starting point. Setting the seed fixes that starting point, so you get the same sequence every time. That matters more than it sounds: it is how machine learning experiments are made reproducible, and you will see `random_state=42` all over scikit-learn for exactly this reason.

In [30]:
import datetime

release = datetime.date(2024, 3, 15)
today = datetime.date(2024, 6, 1)

print("Released:", release.strftime("%d %B %Y"))
print("Days since release:", (today - release).days)

Released: 15 March 2024
Days since release: 78


A rough map of the standard library, grouped by what you would reach for it:

| Need | Modules |
|---|---|
| Files and paths | `os`, `pathlib`, `shutil`, `glob` |
| Data formats | `csv`, `json`, `pickle` |
| Text | `re`, `string`, `textwrap` |
| Dates and time | `datetime`, `time`, `calendar` |
| Maths and numbers | `math`, `statistics`, `random`, `decimal` |
| Data structures | `collections`, `itertools`, `functools` |
| System | `sys`, `subprocess`, `argparse` |
| Internet | `urllib`, `http`, `socket` |

Several of these get a proper treatment in later chapters — `re` in chapter 16, and `collections`, `itertools` and `functools` in chapter 15.

---
# 6. Third-Party Packages and `pip`

The standard library is large, but it does not contain everything. It has no `pandas`, no `requests`, no `scikit-learn`. Those are **third-party packages**: code written by other people and published to the **Python Package Index**, or **PyPI** (pronounced "pie-pea-eye"), at [pypi.org](https://pypi.org).

The tool that fetches them is **`pip`**, and it ships with Python.

These are terminal commands, not Python code — you type them into your terminal or command prompt, not into a `.py` file:

```bash
pip install requests              # install the latest version
pip install requests==2.31.0      # install one exact version
pip install --upgrade requests    # upgrade to the newest
pip uninstall requests            # remove it
pip list                          # everything currently installed
pip show requests                 # details about one package
```

Once installed, a third-party package imports exactly like a standard library one:

```python
import requests                   # no different from "import csv"
```

Python does not care who wrote a module. Standard library, your own file, something you installed — same `import`, same rules.

> **Tip:** inside a Jupyter notebook you can run these with a `%` prefix, as `%pip install requests`. The `%` version installs into the same Python the notebook is using, which avoids a classic confusion where the package installs somewhere your notebook cannot see it.

---
## When an Import Fails

Three quarters of import problems are one of three things, and Python tells you which — if you read the error.

**`ModuleNotFoundError`: Python cannot find the module at all.**

In [31]:
try:
    import panads          # a very common typo
except ModuleNotFoundError as e:
    print(type(e).__name__, "->", e)

ModuleNotFoundError -> No module named 'panads'


Usually one of:

1. **A typo** in the name, as above.
2. **Not installed** — you need `pip install <name>`.
3. **Installed somewhere else** — most often, installed outside the virtual environment you are currently in. This is why the next section matters.

**`ImportError`: the module was found, but the name inside it was not.**

In [32]:
try:
    from math import square_root      # math has sqrt, not square_root
except ImportError as e:
    print(type(e).__name__)
    print("Found the math module, but it has no 'square_root' -- it is called 'sqrt'.")

ImportError
Found the math module, but it has no 'square_root' -- it is called 'sqrt'.


**Making a dependency optional.** Because a failed import is just an exception, you can catch it — which is how libraries offer features that only light up if you have the extra package installed:

In [33]:
try:
    import fancy_plotting_library
    HAS_PLOTTING = True
except ImportError:
    HAS_PLOTTING = False

if HAS_PLOTTING:
    print("Charts enabled.")
else:
    print("Charts disabled. Install with: pip install fancy_plotting_library")

Charts disabled. Install with: pip install fancy_plotting_library


---
# 7. Virtual Environments

Here is the problem that virtual environments solve.

You build a project in March using `requests` version 2.28. In June you start a new project, and it needs version 2.31. You install the new version — and now the March project is running against a library it was never tested with. Multiply that by twenty packages and several projects, and installations start breaking each other.

A **virtual environment** is a private, isolated Python installation for one project. Each project gets its own folder of packages, so nothing leaks between them.

**Creating and activating one:**

```bash
# 1. Create it (this makes a folder called .venv)
python3 -m venv .venv

# 2. Activate it
source .venv/bin/activate          # macOS / Linux
.venv\Scripts\activate             # Windows

# 3. Your prompt now shows (.venv) — installs go here, and only here
pip install requests

# 4. When you are done
deactivate
```

While the environment is active, `pip install` puts packages inside `.venv/`, and `python3` uses only those packages. Nothing touches your system Python or any other project.

## Sharing an environment: `requirements.txt`

An environment is useless if nobody else can recreate it. The convention is a plain text file listing what the project needs.

**To record what you have:**

```bash
pip freeze > requirements.txt
```

That writes a file like:

```
beautifulsoup4==4.12.3
requests==2.31.0
```

**For somebody else (or future you) to recreate it:**

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

Three commands, and they have exactly the environment the project was built against.

**Two rules worth following from day one:**

1. **Commit `requirements.txt`. Never commit `.venv/`.** The file is a few lines; the folder is thousands of files, is specific to your operating system, and is fully rebuildable from the file. Put `.venv/` in your `.gitignore`.
2. **One environment per project.** They are cheap. Delete and rebuild freely — that is the whole point.

> You may also meet **conda**, which does the same job and is common in data science because it can install non-Python dependencies too. The idea is identical: isolated environments, one per project.

---
# 8. How a Real Project Is Laid Out

Putting all of it together, this is roughly what a small, sane Python project looks like:

```
netflix_analysis/
├── .venv/                  <- the virtual environment (never committed)
├── .gitignore              <- lists .venv/, __pycache__/, data files
├── requirements.txt        <- the packages this project needs
├── README.md               <- what this project is and how to run it
├── main.py                 <- the entry point you actually run
├── data/
│   ├── raw/                <- data as downloaded, never edited
│   └── processed/          <- cleaned data your code produced
└── netflix/                <- your own package
    ├── __init__.py
    ├── catalog.py
    └── ratings.py
```

A few things worth noticing, because they are conventions rather than rules Python enforces:

- **`main.py` is the entry point.** It imports from `netflix/` and does the actual work, wrapped in `if __name__ == "__main__":`.
- **`raw/` is never edited.** If your cleaning script has a bug, you want to be able to start again from the original download.
- **Your own package sits beside `main.py`**, so importing it needs no `sys.path` fiddling.

You do not need this structure for a ten-line script. You will want it the moment a project outlives the afternoon you wrote it in.

---
# Real-World Example: Splitting a Script Into a Project

Let us do the refactor that motivates this whole chapter.

Imagine a single file that loads shows, cleans their titles, computes ratings and prints a report. It works, but everything is tangled together. We already have `text_utils.py` and the `netflix` package, so the report file can become short and readable:

In [34]:
%%writefile module_demo/main.py
# The entry point. Notice how little it does itself: it imports the real work.
# No sys.path line needed here: main.py sits in the same folder as the modules
# it imports, and a script's own folder is always searched first.
import text_utils
from netflix import catalog, ratings

SHOWS = [
    {"title": "  stranger   THINGS ", "runtime": 51, "scores": [4.8, 4.9, 4.7]},
    {"title": "the   crown", "runtime": 58, "scores": [4.2, 4.0, 4.3]},
    {"title": "  WEDNESDAY", "runtime": 47, "scores": [3.4, 3.6, 3.2]},
]


def build_report(shows):
    lines = []
    for show in shows:
        title = text_utils.clean_title(show["title"])
        score = ratings.average(show["scores"])
        lines.append(
            f"{title:<20} {catalog.format_runtime(show['runtime']):>6}  "
            f"{score:.2f}  {ratings.label(score)}"
        )
    return lines


if __name__ == "__main__":
    print("NETFLIX CATALOGUE REPORT")
    print("-" * 58)
    for line in build_report(SHOWS):
        print(line)


Writing module_demo/main.py


In [35]:
!python3 module_demo/main.py

NETFLIX CATALOGUE REPORT
----------------------------------------------------------
Stranger Things         51m  4.80  Critically acclaimed
The Crown               58m  4.17  Well received
Wednesday               47m  3.40  Mixed reviews


Four files, each with one job:

- `text_utils.py` knows about text, and nothing about Netflix.
- `netflix/catalog.py` formats runtimes.
- `netflix/ratings.py` scores things.
- `main.py` wires them together and prints.

The payoff is not cleverness, it is **reuse and testability**. `text_utils.clean_title` can now be used by any project you write, and each piece can be checked on its own — which is exactly what chapter 19 will do with real tests.

---
# Quick Reference

**Importing**

| Syntax | Gets you |
|---|---|
| `import math` | The module; use `math.sqrt()` |
| `from math import sqrt` | Just that name; use `sqrt()` |
| `import numpy as np` | The module under a shorter name |
| `from math import *` | Everything — **avoid** |

**The vocabulary**

| Term | Meaning |
|---|---|
| Module | One `.py` file |
| Package | A folder of modules with an `__init__.py` |
| Standard library | Modules that ship with Python |
| PyPI | The public index of third-party packages |

**`pip`**

```bash
pip install <package>            pip list
pip install <package>==<ver>     pip show <package>
pip uninstall <package>          pip freeze > requirements.txt
```

**Virtual environments**

```bash
python3 -m venv .venv                 # create
source .venv/bin/activate             # activate (macOS/Linux)
.venv\Scripts\activate                # activate (Windows)
pip install -r requirements.txt       # restore from file
deactivate                            # leave
```

**The idiom to remember**

```python
def main():
    ...

if __name__ == "__main__":
    main()
```

Runs when the file is executed directly. Silent when the file is imported.

---

**Next:** chapter 12 looks at iterators and generators — how a `for` loop really works, and how to handle data too large to fit in memory.